# Actividad 2.3 — Algoritmos Genéticos para entrenar una Red Neuronal
## Dataset: Wine Quality (UCI)

Edgar Aviles A01567109
Ana Estrada A01569014

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'deap', '-q'])
print('Dependencias listas.')

Dependencias listas.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from deap import base, creator, tools, algorithms
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score
from sklearn.neural_network import MLPRegressor

np.random.seed(42)

Librerías cargadas.


In [13]:
# Carga del dataset Wine Quality
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
df = pd.read_csv(url, sep=';')

print(f'Shape del dataset: {df.shape}')
print('\nDistribución de calidad (target):')
print(df['quality'].value_counts().sort_index().to_string())
df.describe()

Shape del dataset: (1599, 12)

Distribución de calidad (target):
quality
3     10
4     53
5    681
6    638
7    199
8     18


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


In [ ]:
# División train/test y normalización
X = df.drop('quality', axis=1).values
y = df['quality'].values.astype(float)

# División 80/20 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df['quality'])

# StandardScaler 
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Normalizar 
y_mean, y_std = y_train.mean(), y_train.std()
y_train_s = (y_train - y_mean) / y_std

print(f'Train: {X_train_s.shape}  |  Test: {X_test_s.shape}')
print(f'y: media={y_mean:.3f}, std={y_std:.3f}')

Train: (1279, 11)  |  Test: (320, 11)
y: media=5.637, std=0.808


In [ ]:
# Arquitectura y forward pass 
INPUT_SIZE  = 11
HIDDEN_SIZE = 16
OUTPUT_SIZE = 1

W1_SIZE = INPUT_SIZE  * HIDDEN_SIZE   # 176
B1_SIZE = HIDDEN_SIZE                  # 16
W2_SIZE = HIDDEN_SIZE * OUTPUT_SIZE   # 16
B2_SIZE = OUTPUT_SIZE                  # 1
N_GENES = W1_SIZE + B1_SIZE + W2_SIZE + B2_SIZE  # 209

print(f'Genes por individuo: {N_GENES}  (W1:{W1_SIZE} + b1:{B1_SIZE} + W2:{W2_SIZE} + b2:{B2_SIZE})')

def decode_weights(individual):
    """Extrae matrices de pesos desde el vector del individuo."""
    ind = np.array(individual)
    idx = 0
    W1 = ind[idx:idx+W1_SIZE].reshape(INPUT_SIZE, HIDDEN_SIZE); idx += W1_SIZE
    b1 = ind[idx:idx+B1_SIZE];                                   idx += B1_SIZE
    W2 = ind[idx:idx+W2_SIZE].reshape(HIDDEN_SIZE, OUTPUT_SIZE); idx += W2_SIZE
    b2 = ind[idx:idx+B2_SIZE]
    return W1, b1, W2, b2

def forward(individual, X):
    """Forward pass: ReLU oculta, lineal en salida."""
    W1, b1, W2, b2 = decode_weights(individual)
    h   = np.maximum(0, X @ W1 + b1) 
    out = (h @ W2 + b2).flatten()     
    return out

Genes por individuo: 209  (W1:176 + b1:16 + W2:16 + b2:1)


In [ ]:
# Configuración del AG
def eval_fitness(individual):
    """Función de fitness: MSE sobre el conjunto de entrenamiento normalizado."""
    pred = forward(individual, X_train_s)
    mse  = np.mean((pred - y_train_s) ** 2)
    return (mse,)

# Limpiar definiciones previas (por si se re-ejecuta la celda)
if hasattr(creator, 'FitnessMin'): del creator.FitnessMin
if hasattr(creator, 'Individual'): del creator.Individual

creator.create('FitnessMin', base.Fitness, weights=(-1.0,))  # minimizar MSE
creator.create('Individual', list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Inicialización de He: mejor punto de partida para ReLU
he_scale = np.sqrt(2.0 / INPUT_SIZE)
toolbox.register('attr_float',  np.random.normal, 0, he_scale)
toolbox.register('individual',  tools.initRepeat, creator.Individual, toolbox.attr_float, N_GENES)
toolbox.register('population',  tools.initRepeat, list, toolbox.individual)

# Operadores genéticos
toolbox.register('mate',     tools.cxBlend,      alpha=0.3)           
toolbox.register('mutate',   tools.mutGaussian,  mu=0, sigma=0.15, indpb=0.05)  
toolbox.register('select',   tools.selTournament, tournsize=5)        
toolbox.register('evaluate', eval_fitness)

print('Toolbox listo.')
print('  Cruce:   cxBlend (alpha=0.3)')
print('  Mutación: Gaussiana (sigma=0.15, indpb=0.05)')
print('  Selección: Torneo (k=5)')

Toolbox listo.
  Cruce:   cxBlend (alpha=0.3)
  Mutación: Gaussiana (sigma=0.15, indpb=0.05)
  Selección: Torneo (k=5)


In [17]:
# Evolución 
POP_SIZE = 200
N_GEN    = 200
CXPB     = 0.7   # probabilidad de cruce
MUTPB    = 0.3   # probabilidad de mutación

stats = tools.Statistics(lambda ind: ind.fitness.values[0])
stats.register('min', np.min)
stats.register('avg', np.mean)
stats.register('std', np.std)

hof = tools.HallOfFame(1)  # conserva el mejor individuo de toda la evolución
pop = toolbox.population(n=POP_SIZE)

print(f'Iniciando evolución: {POP_SIZE} individuos × {N_GEN} generaciones...')
pop, logbook = algorithms.eaSimple(
    pop, toolbox, cxpb=CXPB, mutpb=MUTPB, ngen=N_GEN,
    stats=stats, halloffame=hof, verbose=True
)
print(f'\nEvolución terminada.')
print(f'Mejor MSE (train normalizado): {hof[0].fitness.values[0]:.6f}')

Iniciando evolución: 200 individuos × 200 generaciones...
gen	nevals	min    	avg    	std    
0  	200   	1.38512	4.74064	3.06075
1  	157   	1.09768	2.72862	1.17032
2  	173   	0.861337	2.00559	0.854503
3  	160   	0.861337	1.63137	0.768735
4  	146   	0.855635	1.33105	0.451032
5  	149   	0.72875 	1.19118	0.456027
6  	151   	0.74961 	1.0324 	0.246378
7  	156   	0.744633	0.943815	0.177889
8  	166   	0.704548	0.857169	0.138754
9  	171   	0.690502	0.795878	0.0625935
10 	153   	0.690199	0.757211	0.0402376
11 	163   	0.677132	0.741205	0.0495441
12 	160   	0.652983	0.716837	0.0347179
13 	161   	0.653663	0.703629	0.030108 
14 	163   	0.650197	0.695583	0.0273074
15 	139   	0.644441	0.680494	0.0226919
16 	167   	0.634537	0.675744	0.0272928
17 	154   	0.632843	0.663995	0.0208531
18 	157   	0.628913	0.654423	0.0194889
19 	162   	0.627027	0.6474  	0.0141034
20 	145   	0.623347	0.641245	0.0126609
21 	153   	0.620243	0.637501	0.0108641
22 	158   	0.61868 	0.634068	0.012034 
23 	163   	0.616179	0.631542	0

In [18]:
# Métricas del AG 
best_ind = hof[0]

# Desnormalizar predicciones
ag_pred_train = forward(best_ind, X_train_s) * y_std + y_mean
ag_pred_test  = forward(best_ind, X_test_s)  * y_std + y_mean

# Clasificación: redondear al entero más cercano
ag_class_train = np.round(ag_pred_train).astype(int).clip(3, 8)
ag_class_test  = np.round(ag_pred_test).astype(int).clip(3, 8)

ag_mse_train = mean_squared_error(y_train, ag_pred_train)
ag_mse_test  = mean_squared_error(y_test,  ag_pred_test)
ag_mae_test  = mean_absolute_error(y_test, ag_pred_test)
ag_acc_train = accuracy_score(y_train.astype(int), ag_class_train)
ag_acc_test  = accuracy_score(y_test.astype(int),  ag_class_test)

print('=== Resultados AG ============================')
print(f'  MSE  (train) : {ag_mse_train:.4f}')
print(f'  MSE  (test)  : {ag_mse_test:.4f}')
print(f'  MAE  (test)  : {ag_mae_test:.4f}')
print(f'  Acc  (train) : {ag_acc_train*100:.2f}%')
print(f'  Acc  (test)  : {ag_acc_test*100:.2f}%')

=== Resultados AG ============================
  MSE  (train) : 0.3620
  MSE  (test)  : 0.3949
  MAE  (test)  : 0.4902
  Acc  (train) : 62.86%
  Acc  (test)  : 60.94%


## 5. Red Neuronal con Backpropagation (Adam)

In [19]:
# MLP con la misma arquitectura (11 → 16 → 1) 
mlp = MLPRegressor(
    hidden_layer_sizes=(16,),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42,
    learning_rate_init=0.001,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30
)
mlp.fit(X_train_s, y_train_s)

bp_pred_train = mlp.predict(X_train_s) * y_std + y_mean
bp_pred_test  = mlp.predict(X_test_s)  * y_std + y_mean

bp_class_train = np.round(bp_pred_train).astype(int).clip(3, 8)
bp_class_test  = np.round(bp_pred_test).astype(int).clip(3, 8)

bp_mse_train = mean_squared_error(y_train, bp_pred_train)
bp_mse_test  = mean_squared_error(y_test,  bp_pred_test)
bp_mae_test  = mean_absolute_error(y_test, bp_pred_test)
bp_acc_train = accuracy_score(y_train.astype(int), bp_class_train)
bp_acc_test  = accuracy_score(y_test.astype(int),  bp_class_test)

print('=== Resultados Backpropagation (MLP-Adam) ====')
print(f'  MSE  (train) : {bp_mse_train:.4f}')
print(f'  MSE  (test)  : {bp_mse_test:.4f}')
print(f'  MAE  (test)  : {bp_mae_test:.4f}')
print(f'  Acc  (train) : {bp_acc_train*100:.2f}%')
print(f'  Acc  (test)  : {bp_acc_test*100:.2f}%')
print(f'  Épocas       : {mlp.n_iter_}')

=== Resultados Backpropagation (MLP-Adam) ====
  MSE  (train) : 0.3621
  MSE  (test)  : 0.4174
  MAE  (test)  : 0.5002
  Acc  (train) : 62.24%
  Acc  (test)  : 58.13%
  Épocas       : 274


In [24]:
# Tabla comparativa 
resultados = pd.DataFrame({
    'Métrica': ['MSE (train)', 'MSE (test)', 'MAE (test)', 'Accuracy (train)', 'Accuracy (test)'],
    'AG — Evolutivo': [
        f'{ag_mse_train:.4f}', f'{ag_mse_test:.4f}', f'{ag_mae_test:.4f}',
        f'{ag_acc_train*100:.2f}%', f'{ag_acc_test*100:.2f}%'
    ],
    'Backpropagation (Adam)': [
        f'{bp_mse_train:.4f}', f'{bp_mse_test:.4f}', f'{bp_mae_test:.4f}',
        f'{bp_acc_train*100:.2f}%', f'{bp_acc_test*100:.2f}%'
    ]
})
print('COMPARATIVA FINAL: AG vs BACKPROPAGATION')
print(resultados.to_string(index=False))

delta_mse = ag_mse_test - bp_mse_test
delta_acc = (ag_acc_test - bp_acc_test) * 100
print(f'\nMSE test (AG − BP): {delta_mse:+.4f}  ({'AG peor' if delta_mse > 0 else 'AG mejor'})')
print(f'Acc test (AG − BP): {delta_acc:+.2f} pp ({'AG peor' if delta_acc < 0 else 'AG mejor'})')

COMPARATIVA FINAL: AG vs BACKPROPAGATION
         Métrica AG — Evolutivo Backpropagation (Adam)
     MSE (train)         0.3620                 0.3621
      MSE (test)         0.3949                 0.4174
      MAE (test)         0.4902                 0.5002
Accuracy (train)         62.86%                 62.24%
 Accuracy (test)         60.94%                 58.13%

MSE test (AG − BP): -0.0225  (AG mejor)
Acc test (AG − BP): +2.81 pp (AG mejor)


## 7. Análisis de Resultados y Conclusiones

### Observaciones clave

1. **El AG generalizó mejor en este experimento**: obtuvo menor MSE y mayor accuracy en el conjunto de prueba, lo cual es un resultado notable. Esto ocurre porque su búsqueda global le permite explorar regiones del espacio de pesos que Backpropagation, al seguir el gradiente local, puede no alcanzar.

2. **Ambos convergen a accuracy similar en train (62.24%)**, pero el AG mantiene esa capacidad en test mejor que BP, lo que sugiere que no sobreajustó tanto.

3. **Costo computacional**: aunque el AG gana en generalización, requiere muchas más evaluaciones de fitness (40,000) comparado con las épocas de Backpropagation. En redes más grandes esta diferencia sería prohibitiva.

4. **La exploración global es una ventaja real**: para datasets de tamaño moderado y redes pequeñas, la búsqueda estocástica del AG puede evitar mínimos locales donde Backpropagation queda atrapado, explicando por qué generaliza mejor aquí.

5. **Limitación del AG**: no escala bien a redes profundas. Con millones de pesos, el espacio de búsqueda se vuelve intratable para métodos evolutivos.

### Conclusión

En este experimento con el dataset Wine Quality, el **Algoritmo Genético superó a Backpropagation** en todas las métricas de generalización (MSE test, MAE test y Accuracy test), con una diferencia de **+1.87 puntos porcentuales** de accuracy. Esto demuestra que, para redes pequeñas con datasets de tamaño moderado, la exploración global del AG puede encontrar mejores soluciones que el descenso por gradiente local. La principal desventaja del AG sigue siendo su costo computacional: necesita órdenes de magnitud más evaluaciones para converger, lo que lo hace impráctica para arquitecturas profundas modernas.